# Patch SAE segmentation: 24-point Kaggle campaign

Enable **Internet**, select **T4 GPU**, and enable a **HF_TOKEN** secret with
read/write access to your Hugging Face **dataset** repository. Push the new
code to GitHub `refactor` before running. Run one encoder/dataset job per
session initially: each job has 3 SAE fits and 4 evaluations. No backbone
fine-tuning, ImageNet download, inference sweep, or figure generation.

The notebook restores and publishes feature shards, training checkpoints,
evaluation progress and tables. A session lost between backups repeats only
work since its last uploaded boundary. Use a private notebook for private results.


In [ ]:
ENCODER = "dinov2"  # "dinov2", "siglip2", "clip"; "all" runs sequentially
DATASET = "voc"     # "voc", "ade20k"; "all" runs sequentially
HF_REPO = "quangnguyenopt/MPSAE"
GIT_REPO = "https://github.com/thienquang204/MPSAE.git"
GIT_BRANCH = "refactor"
PUBLISH = True


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HOME"] = "/tmp/mpsae-hf"
os.environ["HF_DATASETS_CACHE"] = "/tmp/mpsae-datasets"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"
REPO = Path("/kaggle/working/MPSAE")
WORK = Path("/kaggle/working/patch-scene-v2")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, "--depth", "1", GIT_REPO, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
subprocess.run(["nvidia-smi"], check=True)
assert (REPO / "mpsae/patch_scene/__main__.py").exists(), "Push the new code to GitHub first."


## Isolated environment
Uses Python 3.11 and CUDA 12.6 PyTorch, independently of Kaggle's installed
Python/packages. P100 is unsupported by the project guard. T4 x2 uses GPU 0.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
PYTHON = "/tmp/mpsae-env/bin/python"
if not Path(PYTHON).exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", "/tmp/mpsae-env"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "torch==2.9.1", "torchvision==0.24.1",
                "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "-r", "requirements-patch-scene.txt"], check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], check=True)
subprocess.run([PYTHON, "-m", "mpsae.patch_scene", "doctor", "--device", "cuda:0"], check=True)


In [ ]:
CONFIG = "configs/patch_scene_kaggle.yaml"
PLAN = json.loads(subprocess.check_output([
    PYTHON, "-m", "mpsae.patch_scene", "plan", "--config", CONFIG], text=True))
print(json.dumps(PLAN, indent=2))
assert PLAN["total_evaluations"] == 24 and PLAN["inference_k"] == [12]
print("Selected job:", DATASET, ENCODER)


## Run or resume
The same config and repository resume existing work, including on a new session.
This cell performs real training/evaluation and, by default, publishes artifacts.
GPU extraction and retrieval time depend on your assigned hardware. Start with
DINOv2/VOC; complete the other five encoder/dataset combinations afterwards.


In [ ]:
command = [PYTHON, "-u", "-m", "mpsae.patch_scene", "run",
           "--config", CONFIG, "--encoder", ENCODER, "--dataset", DATASET,
           "--workspace", str(WORK), "--device", "cuda:0", "--repo", HF_REPO]
if PUBLISH:
    command.append("--publish")
subprocess.run(command, check=True)


In [ ]:
from IPython.display import FileLink, display
from zipfile import ZipFile, ZIP_DEFLATED

table = WORK / "campaigns" / PLAN["campaign"] / "table"
print((table / "results.tex").read_text())
archive = Path("/kaggle/working/patch-scene-tables.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for name in ("results.csv", "results.tex", "results.json"):
        z.write(table / name, arcname=name)
display(FileLink(str(archive)))
print("Download patch-scene-tables.zip from Kaggle's file browser if the inline link is unavailable.")


To collect tables from all completed jobs without training, use a fresh
workspace and run `python -m mpsae.patch_scene restore-table --repo
quangnguyenopt/MPSAE --workspace /kaggle/working/recovered-patch-tables` with
`/tmp/mpsae-env/bin/python`. Missing table cells are `--`. Saved notebook output
space is limited; feature backups on HF are independent of notebook outputs.
See `docs/patch_scene_kaggle.md` for the exact protocol and recovery boundaries.
